# 📊 08 · Baseline Comparison

> **Chapter 8.** How do three published missense-effect predictors
> score on our exact paralog-disjoint test set?

## The setup

Every baseline below was run on the same 28,098-variant test split
used everywhere else in the project. We don't report their paper's
headline numbers — we run them from scratch on our data so the
comparison is apples-to-apples.

| Baseline | Year | Source | Training data (⚠️ contamination risk) |
|---|---:|---|---|
| SIFT | 2003 | Ensembl VEP REST | Evolutionary conservation, unsupervised |
| PolyPhen-2 | 2010 | Ensembl VEP REST | HumDiv/HumVar (mild ClinVar overlap) |
| AlphaMissense | 2023 | Google Storage TSV, hg38 | 🔴 Calibrated against ClinVar at release — inflated |

Our own XGBoost is included as a reference line in the forest plot.

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
bl = pd.read_csv(REPO / "results/metrics/baselines_comparison.csv")
cov = pd.read_csv(REPO / "results/metrics/baselines_coverage.csv")
split = pd.read_csv(REPO / "results/metrics/xgboost_split_metrics.csv")

clinvar = bl[bl["slice"] == "clinvar_test"].copy()
our_test = split[split["split"] == "test"].iloc[0]

print("ClinVar test — same protocol everywhere:")
print(f"  {'Baseline':<16s}  {'ROC (CI)':<22s}  {'PR (CI)':<22s}  {'Coverage':>8s}")
for _, r in clinvar.iterrows():
    roc = f"{r['roc_auc']:.3f} [{r['roc_auc_ci_lo']:.3f}, {r['roc_auc_ci_hi']:.3f}]"
    pr = f"{r['pr_auc']:.3f} [{r['pr_auc_ci_lo']:.3f}, {r['pr_auc_ci_hi']:.3f}]"
    print(f"  {r['baseline_display_name']:<16s}  {roc:<22s}  {pr:<22s}  {r['coverage']:>7.0%}")
print(f"  {'XGBoost (ours)':<16s}  {our_test['roc_auc']:.3f} (point)          "
      f"{our_test['pr_auc']:.3f} (point)           100%")

## Forest plot

In [ ]:
from IPython.display import Image  # noqa: PLC0415
Image(filename=str(REPO / "results/figures/baselines_forest_plot.png"))

## What each coverage rate tells us

Coverage = fraction of test variants a baseline could actually score.
A baseline with 20 % coverage that claims ROC=0.95 is *not* the same
story as one with 95 % coverage and ROC=0.90 — the sparse baseline is
cherry-picking easy variants.

| Baseline | Test coverage | Why it's less than 100 % |
|---|---:|---|
| SIFT | 96 % | VEP only provides SIFT scores for canonical-transcript missense consequences |
| PolyPhen-2 | 93 % | Same — PolyPhen-2 is not computed for all variants |
| AlphaMissense | 86 % | AlphaMissense only covers canonical UniProt isoforms from the human proteome |
| XGBoost (ours) | 100 % | By construction — every test variant is in the training feature schema |

---

## Contamination caveats baked into the output CSV

Every row of `baselines_comparison.csv` carries a
`training_contamination_warning` column. Reviewers see the caveat in
the data, not only in the prose.

In [ ]:
print(bl[["baseline", "training_data", "training_contamination_warning"]].drop_duplicates("baseline").to_string(index=False, max_colwidth=90))

## The takeaway

1. **Our XGBoost cleanly outperforms the two classical tools**
   (SIFT +0.22 PR, PolyPhen-2 +0.11 PR) under identical evaluation.
2. **We sit 1.8 pp ROC / 5.2 pp PR below AlphaMissense**, under
   stricter methodology (no ClinVar calibration leak).
3. The remaining gap to AlphaMissense is precisely what Phase 2 is
   about — adding orthogonal ESM-2 + AlphaFold2 signal to close it
   without inheriting the ClinVar contamination.

---

## Reproduce

```bash
# Download once (~643 MB, then cached):
curl -o data/raw/baselines/alphamissense/AlphaMissense_hg38.tsv.gz \
  https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz

# Then:
python scripts/run_baselines.py --n-boot 1000 --seed 42
```